# Defend the center: LIF SNN from scratch

Standard `doom_defend_the_center`, not the flat-actions variant. Two categorical
factors control turning (3 choices) and attack (2 choices). Both policies use five
logits, allowing six combinations including turning while shooting.

Use a fresh Linux GPU kernel for each notebook and run from top to bottom.
First run defaults to a 100k-step smoke test with 10 evaluation episodes. For a
full matched comparison set SMOKE_TEST=False in both notebooks (8M steps, 300 episodes).
The SNN does not load ANN weights. Fresh mode refuses existing run directories.
Evaluation uses the full Sample Factory actor-critic with trained normalization;
there is no four-action standalone extraction or inherited basic-scenario score table.


## 1. Server and GPU setup

In [ ]:
import os, sys, json, random, hashlib, time, importlib
from pathlib import Path
from importlib.metadata import version
# This variable also reaches newly spawned Sample Factory checkpoint loaders.
# Only load checkpoints produced by this project or another trusted source.
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import numpy as np
import torch
import gymnasium as gym
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "defend the center").is_dir() and (p / "conversion_friendly_v1").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter from this repository.")
WORK_DIR = ROOT / "defend the center"
os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path: sys.path.insert(0, str(WORK_DIR))
OUTPUT_ROOT = Path(os.environ.get("DEFEND_OUTPUT_ROOT", str(WORK_DIR / "outputs"))).expanduser().resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
assert torch.cuda.is_available(), "Launch this kernel inside a GPU allocation."
DEVICE = "cuda"
CPU_BUDGET = int(os.environ.get("SLURM_CPUS_PER_TASK", len(os.sched_getaffinity(0))))
NUM_WORKERS = max(1, min(8, CPU_BUDGET, int(os.environ.get("DOOM_NUM_WORKERS", max(1, CPU_BUDGET-2)))))
print("Kernel:", sys.executable, "GPU:", torch.cuda.get_device_name(0), "workers:", NUM_WORKERS)
print("GPU test:", (torch.ones(1, device=DEVICE)+1).item())


## 2. Settings

In [ ]:
ENV_NAME = "doom_defend_the_center"
SEED = 0
RUN_MODE = "fresh"          # fresh, resume, evaluate
RUN_TAG = "v1"              # new tag for each independent run
SMOKE_TEST = True           # switch False for full training
TOTAL_STEPS = 100_000 if SMOKE_TEST else 8_000_000
BATCH_SIZE = 1024
LEARNING_RATE = 1e-4
ENVS_PER_WORKER = 4
N_EVAL_EPISODES = 10 if SMOKE_TEST else 300
EVAL_SEEDS = list(range(10000, 10000 + N_EVAL_EPISODES))
assert RUN_MODE in {"fresh", "resume", "evaluate"}
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
SNN_T = 8
LIF_BETA = 0.9
TARGET_RATE = 0.15
FEAT_SCALE = 4.0
EXPERIMENT = f"defend_lif_beta{LIF_BETA:g}_T{SNN_T}_seed{SEED}_{RUN_TAG}" + ("_smoke" if SMOKE_TEST else "")


## 3. Importable LIF implementation

Same LIF trunk, triangular surrogate and subtractive reset as our basic-scenario experiment. Membrane state resets each observation. The standalone helper in this module is not used for this tuple-action evaluator.

In [ ]:
%%writefile defend_lif.py
"""Directly trained spiking policy for Sample Factory (ViZDoom doom_defend_the_center).

Same layer sizes as the ANN baseline (convnet_simple): 3 convs + a 512 layer, but every
hidden layer is an leaky integrate-and-fire (LIF) layer unrolled over T time steps and trained
with surrogate gradients. State is reset for every frame (no recurrence across frames).
"""
import numpy as np
import torch
from torch import nn

from sample_factory.algo.utils.torch_utils import calc_num_elements
from sample_factory.model.encoder import Encoder


# ----------------------------------------------------------------------------------------
# Neuron: Heaviside spike in the forward pass, triangular surrogate in the backward pass
# ----------------------------------------------------------------------------------------
class SpikeFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):  # x = membrane - threshold
        ctx.save_for_backward(x)
        return (x >= 0).to(x.dtype)

    @staticmethod
    def backward(ctx, grad_out):
        (x,) = ctx.saved_tensors
        return grad_out * torch.clamp(1.0 - x.abs(), min=0.0)  # triangular, width 1


def lif_step(v, current, threshold=1.0, v_init=0.5, detach_reset=True, beta=0.9):
    """One LIF step: decay voltage, add current, spike, subtract threshold.

    beta=1 recovers the IF update; current is not multiplied by (1-beta)."""
    if v is None:
        v = torch.full_like(current, v_init * threshold)
    v = beta * v + current
    s = SpikeFn.apply(v - threshold)
    v = v - (s.detach() if detach_reset else s) * threshold
    return v, s


# ----------------------------------------------------------------------------------------
# The spiking trunk (no Sample Factory dependency in the maths, so it can be evaluated alone)
# ----------------------------------------------------------------------------------------
class SpikingTrunk(nn.Module):
    def __init__(self, in_channels=3, obs_hw=(72, 128), T=8, v_init=0.5,
                 gains=(1.0, 1.0, 1.0, 1.0), feat_scale=4.0, beta=0.9):
        super().__init__()
        if not 0.0 <= float(beta) <= 1.0:
            raise ValueError("beta must be in [0, 1]")
        self.register_buffer("beta", torch.tensor(float(beta)))
        self.T = int(T)
        self.v_init = float(v_init)
        self.conv1 = nn.Conv2d(in_channels, 32, 8, stride=4)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2)
        self.conv3 = nn.Conv2d(64, 128, 3, stride=2)
        # local (unregistered) stack, only used to compute the flattened size; keeps state_dict keys unique
        self.flat_size = calc_num_elements(nn.Sequential(self.conv1, self.conv2, self.conv3), (in_channels, *obs_hw))
        self.fc = nn.Linear(self.flat_size, 512)
        # fixed (non-trainable) per-layer current gains and output scale: buffers, so they are saved
        # in the checkpoint and are not touched by Sample Factory's weight initialisation
        self.register_buffer("gains", torch.tensor(list(gains), dtype=torch.float32))
        self.register_buffer("feat_scale", torch.tensor(float(feat_scale)))
        self.out_size = 512

    def forward(self, x, T=None, return_spikes=False):
        T = self.T if T is None else int(T)
        g = self.gains
        c1 = self.conv1(x) * g[0]  # direct encoding: identical every step, computed once
        v1 = v2 = v3 = v4 = None
        acc = 0
        spikes = [0.0, 0.0, 0.0, 0.0]
        for _ in range(T):
            v1, s1 = lif_step(v1, c1, v_init=self.v_init, beta=self.beta)
            v2, s2 = lif_step(v2, self.conv2(s1) * g[1], v_init=self.v_init, beta=self.beta)
            v3, s3 = lif_step(v3, self.conv3(s2) * g[2], v_init=self.v_init, beta=self.beta)
            v4, s4 = lif_step(v4, self.fc(s3.flatten(1)) * g[3], v_init=self.v_init, beta=self.beta)
            acc = acc + s4
            if return_spikes:
                for k, s in enumerate((s1, s2, s3, s4)):
                    spikes[k] = spikes[k] + s.detach().flatten(1).sum(1)  # per-sample spike count
        feats = acc * (self.feat_scale / T)  # mean firing rate of the 512 layer, scaled
        if return_spikes:
            return feats, torch.stack(spikes, 1)  # (B, 4) spikes per decision per layer
        return feats


# ----------------------------------------------------------------------------------------
# Sample Factory glue
# ----------------------------------------------------------------------------------------
def add_snn_args(parser):
    p = parser
    p.add_argument("--snn_beta", default=0.9, type=float, help="Fixed LIF membrane retention in [0, 1]")
    p.add_argument("--snn_T", default=8, type=int, help="Time steps per frame for the spiking encoder")
    p.add_argument("--snn_v_init", default=0.5, type=float, help="Initial membrane potential / threshold")
    p.add_argument("--snn_gains", default="1,1,1,1", type=str,
                   help="Comma separated fixed current gains for the 4 spiking layers (from calibration)")
    p.add_argument("--snn_feat_scale", default=4.0, type=float, help="Scale on the mean firing rate fed to the heads")


class SpikingEncoder(Encoder):
    def __init__(self, cfg, obs_space):
        super().__init__(cfg)
        obs_shape = obs_space["obs"].shape
        gains = [float(v) for v in str(cfg.snn_gains).split(",")]
        assert len(gains) == 4, "--snn_gains needs 4 comma separated numbers"
        self.trunk = SpikingTrunk(obs_shape[0], tuple(obs_shape[1:]), cfg.snn_T, cfg.snn_v_init,
                                  gains, cfg.snn_feat_scale, beta=cfg.snn_beta)
        self.encoder_out_size = self.trunk.out_size

    def forward(self, obs_dict):
        return self.trunk(obs_dict["obs"])

    def get_out_size(self):
        return self.encoder_out_size


def make_snn_encoder(cfg, obs_space):
    return SpikingEncoder(cfg, obs_space)


# ----------------------------------------------------------------------------------------
# Data-dependent gain calibration (so the network does not start silent)
# ----------------------------------------------------------------------------------------
@torch.no_grad()
def calibrate_gains(frames, T=8, target_rate=0.15, v_init=0.5, iters=18, device="cpu", seed=0, beta=0.9):
    """Pick fixed per-layer current gains so each layer's mean firing rate is ~target_rate at
    initialisation, using real (already standardised) frames. Weights use the same init Sample
    Factory applies (orthogonal, gain 1, zero bias)."""
    torch.manual_seed(seed)
    trunk = SpikingTrunk(frames.shape[1], tuple(frames.shape[2:]), T, v_init, beta=beta).to(device)
    for m in (trunk.conv1, trunk.conv2, trunk.conv3, trunk.fc):
        nn.init.orthogonal_(m.weight, gain=1.0)
        nn.init.zeros_(m.bias)
    x = frames.to(device)
    gains = [1.0, 1.0, 1.0, 1.0]

    def rate_of_layer(k, g):
        gg = list(gains)
        gg[k] = g
        c1 = trunk.conv1(x) * gg[0]
        v = [None] * 4
        total, count = 0.0, 0
        for _ in range(T):
            v[0], s1 = lif_step(v[0], c1, v_init=v_init, beta=beta)
            if k == 0:
                total += s1.sum().item(); count += s1.numel(); continue
            v[1], s2 = lif_step(v[1], trunk.conv2(s1) * gg[1], v_init=v_init, beta=beta)
            if k == 1:
                total += s2.sum().item(); count += s2.numel(); continue
            v[2], s3 = lif_step(v[2], trunk.conv3(s2) * gg[2], v_init=v_init, beta=beta)
            if k == 2:
                total += s3.sum().item(); count += s3.numel(); continue
            v[3], s4 = lif_step(v[3], trunk.fc(s3.flatten(1)) * gg[3], v_init=v_init, beta=beta)
            total += s4.sum().item(); count += s4.numel()
        return total / count

    for k in range(4):
        lo, hi = -4.0, 4.0  # search log10(gain)
        for _ in range(iters):
            mid = 0.5 * (lo + hi)
            if rate_of_layer(k, 10 ** mid) < target_rate:
                lo = mid
            else:
                hi = mid
        gains[k] = float(10 ** (0.5 * (lo + hi)))
    rates = [rate_of_layer(k, gains[k]) for k in range(4)]
    return gains, rates


# ----------------------------------------------------------------------------------------
# Standalone evaluator (no Sample Factory model), mirrors the PlainCNN / converted-SNN protocol
# ----------------------------------------------------------------------------------------
class DirectSNN(nn.Module):
    def __init__(self, T=8, v_init=0.5, gains=(1, 1, 1, 1), feat_scale=4.0, n_actions=4, beta=0.9):
        super().__init__()
        self.trunk = SpikingTrunk(3, (72, 128), T, v_init, gains, feat_scale, beta=beta)
        self.head = nn.Linear(512, n_actions)

    def forward(self, x, T=None, return_spikes=False):
        out = self.trunk(x, T=T, return_spikes=return_spikes)
        if return_spikes:
            feats, spk = out
            return self.head(feats), spk
        return self.head(out)


def load_direct_snn(sf_state_dict, T, v_init=0.5, n_actions=4):
    """Build the standalone SNN from a Sample Factory checkpoint['model'] state dict."""
    gains = sf_state_dict["encoder.trunk.gains"].cpu().tolist()
    feat_scale = float(sf_state_dict["encoder.trunk.feat_scale"])
    beta = float(sf_state_dict["encoder.trunk.beta"].cpu())
    net = DirectSNN(T, v_init, gains, feat_scale, n_actions, beta=beta)
    state = {}
    for k, v in sf_state_dict.items():
        if k.startswith("encoder.trunk."):
            state["trunk." + k[len("encoder.trunk."):]] = v
    state["head.weight"] = sf_state_dict["action_parameterization.distribution_linear.weight"]
    state["head.bias"] = sf_state_dict["action_parameterization.distribution_linear.bias"]
    missing, unexpected = net.load_state_dict(state, strict=False)
    return net.eval(), missing, unexpected

## Register the policy and check the environment

In [ ]:
import functools
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args, load_from_checkpoint
from sample_factory.envs.create_env import create_env
from sample_factory.envs.env_utils import register_env
from sample_factory.model.actor_critic import create_actor_critic
from sample_factory.algo.utils.rl_utils import prepare_and_normalize_obs
from sample_factory.train import run_rl
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec

for spec in DOOM_ENVS:
    register_env(spec.name, functools.partial(make_doom_env_from_spec, spec))
import defend_lif as sd
importlib.reload(sd)
global_model_factory().register_encoder_factory(sd.make_snn_encoder)

def parse_cfg(argv, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser); doom_override_defaults(parser)
    sd.add_snn_args(parser)
    return parse_full_cfg(parser, argv)

TRAIN_DIR = str(OUTPUT_ROOT / "train_dir")
RUN_DIR = Path(TRAIN_DIR) / EXPERIMENT
if RUN_MODE == "fresh" and RUN_DIR.exists():
    raise FileExistsError(f"Fresh run refuses to overwrite/resume {RUN_DIR}. Choose a new RUN_TAG.")
if RUN_MODE != "fresh" and not (RUN_DIR / "config.json").is_file():
    raise FileNotFoundError(f"No saved configuration in {RUN_DIR}")
env_cfg = parse_cfg([f"--env={ENV_NAME}", "--experiment=defend_probe",
                     f"--train_dir={TRAIN_DIR}", f"--seed={SEED}",
                     "--res_h=72", "--res_w=128", "--env_frameskip=4"], evaluation=True)
env = create_env(ENV_NAME, cfg=env_cfg, env_config=None)
assert isinstance(env.action_space, gym.spaces.Tuple)
assert [s.n for s in env.action_space.spaces] == [3, 2], env.action_space
ACTION_SPACE = env.action_space
raw_space = env.observation_space
OBS_SPACE = raw_space if isinstance(raw_space, gym.spaces.Dict) else gym.spaces.Dict({"obs": raw_space})
assert OBS_SPACE["obs"].shape == (3, 72, 128)
obs, _ = env.reset(seed=SEED)
env.step(ACTION_SPACE.sample())
env.close()
print("Observation:", OBS_SPACE, "Action:", ACTION_SPACE)
print("Turning: idle/left/right; attack: off/on. Five logits, six combinations.")


## Calibrate only for fresh training

Use defend-the-center images. Resume/evaluate reads original gains instead of recalibrating. Calibration is not ANN pretraining.

In [ ]:
if RUN_MODE == "fresh":
    env = create_env(ENV_NAME, cfg=env_cfg, env_config=None)
    env.action_space.seed(SEED)
    raw_frames = []; episode = 0
    try:
        while len(raw_frames) < 800:
            obs, _ = env.reset(seed=SEED + episode); episode += 1
            done = truncated = False; t = 0
            while not (done or truncated) and len(raw_frames) < 800:
                raw = obs["obs"] if isinstance(obs, dict) else obs
                if t % 2 == 0: raw_frames.append(np.asarray(raw).copy())
                obs, _, done, truncated, _ = env.step(env.action_space.sample()); t += 1
    finally:
        env.close()
    x = torch.tensor(np.stack(raw_frames)).float() / 255.0
    x = ((x-x.mean(0, keepdim=True))/torch.sqrt(x.var(0, unbiased=False, keepdim=True)+1e-5)).clamp(-5,5)
    gains, rates = sd.calibrate_gains(x, T=SNN_T, target_rate=TARGET_RATE,
                                     device=DEVICE, seed=SEED, beta=LIF_BETA)
    path = OUTPUT_ROOT / "calibration" / EXPERIMENT
    path.mkdir(parents=True, exist_ok=True)
    torch.save({"frames": x, "gains": gains, "rates": rates, "seed": SEED,
                "beta": LIF_BETA, "T": SNN_T}, path / "initialization.pt")
    print("Gains:", gains, "rates:", rates)
else:
    # Never recalibrate a resumed model: read its original settings.
    saved = json.loads((RUN_DIR / "config.json").read_text())
    assert saved["env"] == ENV_NAME
    gains = [float(v) for v in saved["snn_gains"].split(",")]
    SNN_T = int(saved["snn_T"]); LIF_BETA = float(saved["snn_beta"])
    FEAT_SCALE = float(saved["snn_feat_scale"])
    print("Using saved LIF settings and gains.")


## Train with Sample Factory

PPO trains the policy and critic from rewards. Fresh runs initialize weights; resume uses this experiment's saved state. Check the resolved config/logs: resumed settings can differ from requested arguments.

In [ ]:
argv = [f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}",
        f"--seed={SEED}", "--device=gpu", "--use_rnn=False", "--nonlinearity=relu",
        "--actor_critic_share_weights=True", "--res_h=72", "--res_w=128", "--env_frameskip=4",
        f"--num_workers={NUM_WORKERS}", f"--num_envs_per_worker={ENVS_PER_WORKER}",
        f"--batch_size={BATCH_SIZE}", f"--learning_rate={LEARNING_RATE}",
        f"--train_for_env_steps={TOTAL_STEPS}", "--set_workers_cpu_affinity=False"]
argv += [f"--snn_T={SNN_T}", f"--snn_beta={LIF_BETA}",
         "--snn_gains=" + ",".join(f"{g:.6f}" for g in gains), f"--snn_feat_scale={FEAT_SCALE}"]
if RUN_MODE != "evaluate":
    cfg = parse_cfg(argv)
    if RUN_MODE == "fresh" and RUN_DIR.exists():
        raise FileExistsError("Run directory appeared; use a new tag or deliberately select resume.")
    t0 = time.time()
    status = run_rl(cfg)
    print("Status:", status, "minutes:", (time.time()-t0)/60)
    assert status == 0, "Training failed; inspect logs before evaluation."
else:
    print("Training skipped: evaluation mode")


## Reload the latest checkpoint

In [ ]:
checkpoints = sorted((RUN_DIR / "checkpoint_p0").glob("checkpoint_*.pth"))
if not checkpoints: raise FileNotFoundError(f"No checkpoints in {RUN_DIR}")
CKPT_PATH = checkpoints[-1]  # latest regular checkpoint, not best-reward checkpoint
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
eval_cfg = parse_cfg([f"--env={ENV_NAME}", f"--experiment={EXPERIMENT}", f"--train_dir={TRAIN_DIR}"], evaluation=True)
eval_cfg = load_from_checkpoint(eval_cfg)
assert eval_cfg.env == ENV_NAME and not eval_cfg.use_rnn
ac = create_actor_critic(eval_cfg, OBS_SPACE, ACTION_SPACE)
ac.load_state_dict(ckpt["model"], strict=True)
ac.model_to_device(torch.device(DEVICE)); ac.eval()

def policy_logits(raw_obs):
    raw_dict = raw_obs if isinstance(raw_obs, dict) else {"obs": raw_obs}
    batch = {k: torch.as_tensor(v).unsqueeze(0) for k,v in raw_dict.items()}
    with torch.no_grad():
        normalized = prepare_and_normalize_obs(ac, batch)
        head = ac.forward_head(normalized)
        core, _ = ac.forward_core(head, None)
        return ac.forward_tail(core, values_only=False, sample_actions=False)["action_logits"]

def tuple_action(logits):
    assert logits.shape == (1, 5)
    return (int(logits[0,:3].argmax()), int(logits[0,3:].argmax()))

# Check both factors independently, including turn-and-shoot combinations.
assert tuple_action(torch.tensor([[0.,2.,1.,0.,3.]])) == (1,1)
print("Loaded:", CKPT_PATH.name, "steps:", ckpt["env_steps"])


## Per-layer firing rates and spike sparsity

Measures the latest loaded checkpoint at its saved training T on up to 500 gameplay observations. Reports fraction of zero spike events, not fraction of permanently silent neurons. Uses separate diagnostic seeds; gameplay/extra inference adds runtime. The replay below still selects the best training checkpoint separately.


In [ ]:
# Measure activity on gameplay observations using this notebook's loaded checkpoint.
# Rate = spikes / (observations * neurons * T), not raw spike count per layer.
ACTIVITY_MAX_OBSERVATIONS = 500
ACTIVITY_MAX_EPISODES = 10
ACTIVITY_FIRST_SEED = 30000
trunk = ac.encoder.trunk
activity_T = int(trunk.T)
activity_beta = float(trunk.beta.detach().cpu())
layer_names = ["conv1","conv2","conv3","fc"]
with torch.no_grad():
    z = torch.zeros((1,3,72,128),device=DEVICE)
    z1 = trunk.conv1(z); z2 = trunk.conv2(z1); z3 = trunk.conv3(z2)
    neurons = np.array([z1[0].numel(),z2[0].numel(),z3[0].numel(),trunk.fc.out_features])
spike_totals = np.zeros(4,dtype=np.float64)
observations = 0
activity_env = create_env(ENV_NAME,cfg=eval_cfg,env_config=None)
try:
    for ep_seed in range(ACTIVITY_FIRST_SEED,ACTIVITY_FIRST_SEED+ACTIVITY_MAX_EPISODES):
        obs,_ = activity_env.reset(seed=ep_seed)
        done = truncated = False
        while not (done or truncated) and observations < ACTIVITY_MAX_OBSERVATIONS:
            raw_dict = obs if isinstance(obs,dict) else {"obs":obs}
            batch = {k:torch.as_tensor(v).unsqueeze(0) for k,v in raw_dict.items()}
            with torch.no_grad():
                normalized = prepare_and_normalize_obs(ac,batch)
                # The same normalized image, dynamics and T as the policy inference.
                _,spikes = trunk(normalized["obs"],return_spikes=True)
                core,_ = ac.forward_core(ac.forward_head(normalized),None)
                logits = ac.forward_tail(core,values_only=False,sample_actions=False)["action_logits"]
            spike_totals += spikes.sum(0).detach().cpu().numpy()
            observations += int(spikes.shape[0])
            obs,_,done,truncated,_ = activity_env.step(tuple_action(logits))
        if observations >= ACTIVITY_MAX_OBSERVATIONS: break
finally:
    activity_env.close()
if observations == 0: raise RuntimeError("No activity observations collected")
rates = spike_totals / (observations * neurons * activity_T)
assert np.all((rates >= 0) & (rates <= 1+1e-6))
activity_rows = []
print(f"Checkpoint: {CKPT_PATH.name}; T={activity_T}; beta={activity_beta:.4f}; observations={observations}")
print(f"{'Layer':8s} {'Neurons':>9s} {'Spikes/obs':>12s} {'Firing rate':>12s} {'Zero events':>12s}")
for k,name in enumerate(layer_names):
    row = {"layer":name,"neurons":int(neurons[k]),"total_spikes":float(spike_totals[k]),
           "spikes_per_observation":float(spike_totals[k]/observations),
           "firing_rate":float(rates[k]),"zero_spike_fraction":float(1-rates[k])}
    activity_rows.append(row)
    print(f"{name:8s} {neurons[k]:9d} {row['spikes_per_observation']:12.1f} {rates[k]:12.2%} {1-rates[k]:12.2%}")
print("Zero-event sparsity counts silent neuron-time slots, not neurons that never fired.")
activity_report = {"checkpoint":str(CKPT_PATH),"T":activity_T,"beta":activity_beta,
                   "observations":observations,"first_seed":ACTIVITY_FIRST_SEED,
                   "max_episodes":ACTIVITY_MAX_EPISODES,"layers":activity_rows}
activity_dir = OUTPUT_ROOT / "results" / EXPERIMENT
activity_dir.mkdir(parents=True,exist_ok=True)
(activity_dir / "lif_activity.json").write_text(json.dumps(activity_report,indent=2))


## Fixed-seed gameplay evaluation

Return and decisions survived are recorded. Decisions are not raw game tics (frame skip is 4). No doom_basic thresholds or kill counts inferred from rewards. These seeds are for validation; use a separate range for final held-out reporting.

In [ ]:
# Deterministic actions on the identical episode-seed list in both notebooks.
env = create_env(ENV_NAME, cfg=eval_cfg, env_config=None)
records = []
try:
    for ep_seed in EVAL_SEEDS:
        obs, _ = env.reset(seed=ep_seed)
        done = truncated = False; total = 0.; decisions = 0
        while not (done or truncated):
            action = tuple_action(policy_logits(obs))
            obs, reward, done, truncated, info = env.step(action)
            total += float(reward); decisions += 1
        records.append({"seed": ep_seed, "return": total, "decisions": decisions,
                        "terminated": bool(done), "truncated": bool(truncated)})
        if len(records) % 10 == 0: print("Episodes:", len(records))
finally:
    env.close()
rewards = np.array([r["return"] for r in records])
print("Mean return:", rewards.mean(), "std:", rewards.std(), "SE:", rewards.std()/np.sqrt(len(rewards)))
print("Mean decisions survived:", np.mean([r["decisions"] for r in records]))
results_dir = OUTPUT_ROOT / "results" / EXPERIMENT
results_dir.mkdir(parents=True, exist_ok=True)
report = {"experiment": EXPERIMENT, "env": ENV_NAME, "checkpoint": str(CKPT_PATH),
          "checkpoint_sha256": hashlib.file_digest(CKPT_PATH.open("rb"), "sha256").hexdigest(),
          "env_steps": int(ckpt["env_steps"]), "mean_return": float(rewards.mean()),
          "std_return": float(rewards.std()), "episodes": records,
          "packages": {p: version(p) for p in ["torch","numpy","sample-factory","vizdoom","gymnasium"]},
          "python": sys.version, "gpu": torch.cuda.get_device_name(0),
          "resolved_training_config": json.loads((RUN_DIR / "config.json").read_text())}
(results_dir / "evaluation.json").write_text(json.dumps(report, indent=2))
np.save(results_dir / "rewards.npy", rewards)
print("Saved:", results_dir)


## Interpretation and limitations

Compare ANN and LIF at equal training budgets using the same evaluation seeds, and repeat training across several seeds.
The environment's return scale differs from doom_basic. A 100k smoke test checks execution, not convergence.
This notebook does not convert the ANN, estimate energy, or extract a standalone policy. The full checkpoint
preserves weights, observation statistics and the critic for future use. Preserve config.json alongside it.
GPU/package compatibility and complete gameplay training still need validation on the university server.
Seeding does not guarantee identical asynchronous training across processes or software versions.


## Replay the best training checkpoint
Runs three deterministic gameplay episodes and embeds MP4 videos. Uses `best_*.pth`,
separately from the latest-checkpoint evaluation above. A short smoke test might not
produce a best checkpoint yet. ffmpeg with libx264 support is required. Frames stream
to the encoder instead of accumulating a long video in RAM. Paths with spaces are supported.
Replays show the best recorded training policy; they do not establish which checkpoint
generalizes best. Rerunning the cell overwrites these seed-specific videos.


In [ ]:
# Replay the checkpoint with the highest recorded training reward.
# This is Sample Factory's best snapshot, not a held-out ranking of every model.
import re, shutil, subprocess, copy
from IPython.display import Video, display

REPLAY_EPISODES = 3
REPLAY_FIRST_SEED = 20000
ffmpeg = shutil.which("ffmpeg")
if ffmpeg is None:
    raise RuntimeError("Replay needs ffmpeg. Load your cluster's ffmpeg module, then rerun this cell.")
best_paths = list((RUN_DIR / "checkpoint_p0").glob("best_*.pth"))
if not best_paths:
    raise FileNotFoundError("No best-reward checkpoint saved yet. Continue training; replay does not silently use latest.")
def recorded_reward(path):
    match = re.search(r"_reward_([-+0-9.eE]+)\.pth$", path.name)
    if match is None:
        raise ValueError(f"Cannot read best reward from {path.name}")
    return float(match.group(1)), path.name
BEST_PATH = max(best_paths, key=recorded_reward)
best_ckpt = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
replay_model = create_actor_critic(eval_cfg, OBS_SPACE, ACTION_SPACE)
replay_model.load_state_dict(best_ckpt["model"], strict=True)
replay_model.model_to_device(torch.device(DEVICE)); replay_model.eval()
replay_cfg = copy.deepcopy(eval_cfg)
replay_env = create_env(ENV_NAME, cfg=replay_cfg, env_config=None, render_mode="rgb_array")
replay_dir = OUTPUT_ROOT / "replays" / EXPERIMENT
replay_dir.mkdir(parents=True, exist_ok=True)
replay_records = []
try:
    for episode in range(REPLAY_EPISODES):
        seed = REPLAY_FIRST_SEED + episode
        video_path = replay_dir / f"best_seed{seed}.mp4"
        log_path = replay_dir / f"best_seed{seed}_ffmpeg.log"
        obs, _ = replay_env.reset(seed=seed)
        done = truncated = False; reward_sum = 0.; frames_written = 0
        encoder = None; log_file = log_path.open("wb")
        try:
            while not (done or truncated):
                frame = replay_env.render()
                if frame is None: raise RuntimeError("rgb_array rendering returned no frame")
                frame = np.asarray(frame)
                if frame.ndim == 3 and frame.shape[0] == 3 and frame.shape[-1] != 3:
                    frame = frame.transpose(1, 2, 0)
                if frame.ndim != 3 or frame.shape[-1] != 3:
                    raise ValueError(f"Unexpected replay frame shape {frame.shape}")
                frame = np.ascontiguousarray(frame, dtype=np.uint8)
                if encoder is None:
                    height, width = frame.shape[:2]
                    # One captured image per environment decision; Doom runs at 35 tics/s.
                    fps = 35.0 / replay_cfg.env_frameskip
                    encoder = subprocess.Popen([
                        ffmpeg, "-y", "-f", "rawvideo", "-pix_fmt", "rgb24",
                        "-s", f"{width}x{height}", "-r", str(fps), "-i", "pipe:0",
                        "-an", "-vf", "pad=ceil(iw/2)*2:ceil(ih/2)*2",
                        "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
                        str(video_path)], stdin=subprocess.PIPE, stdout=subprocess.DEVNULL, stderr=log_file)
                encoder.stdin.write(frame.tobytes()); frames_written += 1
                raw_dict = obs if isinstance(obs, dict) else {"obs": obs}
                batch = {k: torch.as_tensor(v).unsqueeze(0) for k,v in raw_dict.items()}
                with torch.no_grad():
                    normalized = prepare_and_normalize_obs(replay_model, batch)
                    core, _ = replay_model.forward_core(replay_model.forward_head(normalized), None)
                    logits = replay_model.forward_tail(core, values_only=False, sample_actions=False)["action_logits"]
                obs, reward, done, truncated, _ = replay_env.step(tuple_action(logits))
                reward_sum += float(reward)
        finally:
            try:
                if encoder is not None:
                    try:
                        encoder.stdin.close()
                    finally:
                        try: encoder.wait(timeout=60)
                        except subprocess.TimeoutExpired:
                            encoder.kill(); encoder.wait()
                            raise RuntimeError(f"ffmpeg stalled; see {log_path}")
            finally:
                log_file.close()
        if encoder is None or encoder.returncode != 0 or not video_path.is_file() or video_path.stat().st_size == 0:
            raise RuntimeError(f"Video encoding failed; inspect {log_path}")
        replay_records.append({"seed": seed, "return": reward_sum, "frames": frames_written,
                               "video": str(video_path)})
        print("Replay:", video_path, "return:", reward_sum)
        display(Video(str(video_path), embed=True, width=640))
finally:
    replay_env.close()
    del replay_model
    torch.cuda.empty_cache()
with BEST_PATH.open("rb") as checkpoint_file:
    checkpoint_hash = hashlib.file_digest(checkpoint_file, "sha256").hexdigest()
(replay_dir / "replay_metadata.json").write_text(json.dumps({
    "checkpoint": str(BEST_PATH), "checkpoint_sha256": checkpoint_hash,
    "env_steps": int(best_ckpt["env_steps"]), "selection": "highest recorded training reward",
    "frame_skip": replay_cfg.env_frameskip, "episodes": replay_records}, indent=2))
